In [1]:
!pip install datasets   # Load dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [3]:
print(dataset["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [4]:
import pandas as pd

train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print(train_df.head())

                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
2  If only to avoid making this type of film in t...      0
3  This film was probably inspired by Godard's Ma...      0
4  Oh, brother...after hearing about this ridicul...      0


In [5]:
print("Train size:", len(train_df))
print("Test size:", len(test_df))

print("\nTrain label dağılımı:")
print(train_df["label"].value_counts())

Train size: 25000
Test size: 25000

Train label dağılımı:
label
0    12500
1    12500
Name: count, dtype: int64


In [6]:
train_df = train_df.sample(5000, random_state=42)
test_df = test_df.sample(1000, random_state=42)

print("Yeni train size:", len(train_df))
print("Yeni test size:", len(test_df))

Yeni train size: 5000
Yeni test size: 1000


In [7]:
import re

def clean_text(text):
    text = text.lower()                        # küçük harfe çevir
    text = re.sub(r'<.*?>', '', text)          # <br /> gibi HTML tagları sil
    text = re.sub(r'[^a-z0-9\s]', '', text)   # noktalama işaretlerini sil
    text = re.sub(r'\s+', ' ', text).strip()  # fazla boşlukları sil
    return text

train_df['text'] = train_df['text'].apply(clean_text)
test_df['text'] = test_df['text'].apply(clean_text)

print("Temizleme tamamlandı!")
print("\nÖrnek temizlenmiş metin:")
print(train_df['text'].iloc[0][:300])

Temizleme tamamlandı!

Örnek temizlenmiş metin:
dumb is as dumb does in this thoroughly uninteresting supposed black comedy essentially what starts out as chris klein trying to maintain a low profile eventually morphs into an uninspired version of the three amigos only without any laughs in order for black comedy to work it must be outrageous whi


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])

y_train = train_df["label"]
y_test = test_df["label"]

print("Geliştirilmiş TF-IDF tamamlandı")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Geliştirilmiş TF-IDF tamamlandı
X_train shape: (5000, 10000)
X_test shape: (1000, 10000)


In [9]:
import torch

# sparse matrix → normal array → tensor
X_train_tensor = torch.tensor(X_train.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.toarray(), dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

print("Tensor dönüşümü tamam")
print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

Tensor dönüşümü tamam
X_train_tensor: torch.Size([5000, 10000])
y_train_tensor: torch.Size([5000, 1])


In [10]:
import torch.nn as nn

class SentimentNet(nn.Module):
    def __init__(self, input_size):
        super(SentimentNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.sigmoid(x)
        return x

input_size = X_train_tensor.shape[1]
model = SentimentNet(input_size)

print(model)

SentimentNet(
  (fc1): Linear(in_features=10000, out_features=256, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [11]:
import torch.optim as optim

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Hazır")

Hazır


In [26]:
# Modeli sıfırla
model = SentimentNet(input_size)
optimizer = optim.Adam(model.parameters(), lr=0.001)

from torch.utils.data import DataLoader, TensorDataset

# Dataset ve DataLoader oluştur
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

Epoch 1/5, Loss: 0.4519
Epoch 2/5, Loss: 0.1025
Epoch 3/5, Loss: 0.0191
Epoch 4/5, Loss: 0.0029
Epoch 5/5, Loss: 0.0008


In [13]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    predictions = (test_outputs >= 0.5).float()

accuracy = accuracy_score(y_test_tensor, predictions)
print("Test Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test_tensor, predictions))

Test Accuracy: 0.829

Classification Report:
              precision    recall  f1-score   support

         0.0       0.82      0.85      0.84       511
         1.0       0.84      0.81      0.82       489

    accuracy                           0.83      1000
   macro avg       0.83      0.83      0.83      1000
weighted avg       0.83      0.83      0.83      1000



In [14]:
!pip install gradio

In [15]:
import gradio as gr

def predict_sentiment(text):
    # Temizle
    cleaned = clean_text(text)

    # TF-IDF'e çevir
    vectorized = vectorizer.transform([cleaned])
    tensor = torch.tensor(vectorized.toarray(), dtype=torch.float32)

    # Tahmin yap
    model.eval()
    with torch.no_grad():
        output = model(tensor)
        probability = output.item()
        prediction = "Positive 😊" if probability >= 0.5 else "Negative 😞"
        confidence = probability if probability >= 0.5 else 1 - probability

    return f"{prediction} (Confidence: {confidence:.1%})"

# Gradio arayüzü
app = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Write a movie review here...",
        label="Movie Review"
    ),
    outputs=gr.Textbox(label="Sentiment"),
    title="🎬 Sentiment Analysis App",
    description="Enter a movie review and the model will predict if it's Positive or Negative."
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c022e3a97ab73ded0e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
!pip install transformers

In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

# Tokenizer ve model yükle
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

print("Tokenizer yüklendi!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer yüklendi!


In [18]:
# Eğitim ve test metinlerini tokenize et
def tokenize_data(texts, labels, max_length=256):
    encodings = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encodings, torch.tensor(list(labels), dtype=torch.long)

train_encodings, train_labels = tokenize_data(train_df["text"], train_df["label"])
test_encodings, test_labels = tokenize_data(test_df["text"], test_df["label"])

print("Tokenization tamamlandı!")
print("Train shape:", train_encodings["input_ids"].shape)
print("Test shape:", test_encodings["input_ids"].shape)

Tokenization tamamlandı!
Train shape: torch.Size([5000, 256])
Test shape: torch.Size([1000, 256])


In [19]:
from torch.utils.data import DataLoader, TensorDataset

# Dataset oluştur
train_dataset = TensorDataset(
    train_encodings["input_ids"],
    train_encodings["attention_mask"],
    train_labels
)

test_dataset = TensorDataset(
    test_encodings["input_ids"],
    test_encodings["attention_mask"],
    test_labels
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print("DataLoader hazır!")
print("Train batch sayısı:", len(train_loader))

DataLoader hazır!
Train batch sayısı: 313


In [20]:
from transformers import AutoModelForSequenceClassification

# DistilBERT modelini yükle
distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# GPU'ya taşı
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
distilbert_model = distilbert_model.to(device)

print("Model yüklendi!")
print("Kullanılan cihaz:", device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model yüklendi!
Kullanılan cihaz: cuda


In [21]:
from torch.optim import AdamW

optimizer = AdamW(distilbert_model.parameters(), lr=2e-5)

num_epochs = 3

for epoch in range(num_epochs):
    distilbert_model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]

        optimizer.zero_grad()
        outputs = distilbert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

print("Eğitim tamamlandı!")

Epoch 1/3, Loss: 0.3912
Epoch 2/3, Loss: 0.1994
Epoch 3/3, Loss: 0.0941
Eğitim tamamlandı!


In [22]:
from sklearn.metrics import accuracy_score, classification_report

distilbert_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = distilbert_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print("DistilBERT Test Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

DistilBERT Test Accuracy: 0.891

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.85      0.89       511
           1       0.86      0.93      0.89       489

    accuracy                           0.89      1000
   macro avg       0.89      0.89      0.89      1000
weighted avg       0.89      0.89      0.89      1000



In [23]:
import gradio as gr

def predict_distilbert(text):
    cleaned = clean_text(text)
    inputs = tokenizer(
        cleaned,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    distilbert_model.eval()
    with torch.no_grad():
        outputs = distilbert_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()

    label = "Positive 😊" if pred == 1 else "Negative 😞"
    return f"{label} (Confidence: {confidence:.1%})"

app2 = gr.Interface(
    fn=predict_distilbert,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Write a movie review here...",
        label="Movie Review"
    ),
    outputs=gr.Textbox(label="Sentiment"),
    title="🎬 Sentiment Analysis - DistilBERT",
    description="Advanced transformer-based sentiment analysis. Enter a movie review to classify it."
)

app2.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f7c878ab7d1e111b5d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
